In [26]:
import numpy as np
from abc import ABC,abstractmethod
from matplotlib import pyplot as plt
from RBF_BITMAP import rbf_bitmap


**正向传播**
$$
\left(
\begin{matrix}
x_{1}&x_{2}&x_{3}\\
\end{matrix}
\right)

\left(
\begin{matrix}
w_{1}&w_{4} \\
w_{2}&w_{5} \\
w_{3}&w_{6}
\end{matrix}
\right)
+
\left(
\begin{matrix}
b_{1}&b_{2}
\end{matrix}
\right
)
=
\left(
\begin{matrix}
y_{1}&y_{2}
\end{matrix}
\right)
$$

**反向传播**
$$
\begin{align}
&dZ=\Delta_{in}⊙f'(Z_{i}) \\
&dW=\frac{1}{batch} X^T \cdot dZ \\
&db=\frac{1}{batch}\sum_{i=1}^{batch}dZ_{i}
\end{align}
$$
 

In [33]:
class activation(ABC):
    @abstractmethod
    def gradient(self,x)->np.ndarray:
        ...
    @abstractmethod
    def __call__(self,x)->np.ndarray:
        ...
        
class layer(ABC):
    @abstractmethod
    def forward(self,vector_input:np.ndarray)->np.ndarray:
        ...
    @abstractmethod
    def backward(self,grad_input:np.ndarray,learning_rate:float)->np.ndarray:
        ...

In [27]:
class Sigmoid(activation):
    def __call__(self, x:np.ndarray|np.matrix):
        return 1 / (1 + np.exp(-x))

    def gradient(self, x:np.ndarray|np.matrix):
        return self.__call__(x) * (1 - self.__call__(x))
        
class Tanh(activation):
    def __call__(self, x:np.ndarray|np.matrix):
        return 2 / (1 + np.exp(-2*x)) - 1

    def gradient(self, x:np.ndarray|np.matrix):
        return 1 - np.power(self.__call__(x), 2)
        
class Lecun_Tanh(activation):
    def __call__(self, x:np.ndarray|np.matrix):
        return 1.7159 * np.tanh(2/3 * x)

    def gradient(self, x:np.ndarray|np.matrix):
        return 1.14393 * (1 - np.power(np.tanh(2/3 * x), 2))

In [34]:
class Fc_layer(layer):
    m_activation_func:activation=Lecun_Tanh()
    m_batch_size=1
    m_W:np.ndarray
    m_b:np.ndarray
    _forward_in:np.ndarray
    _forward_Z:np.ndarray #线性层输出
    _forward_A:np.ndarray #激活层输出
    m_shape:tuple
    # shape(in_dim,out_dim)


    def __init__(self,shape:tuple,batch_size:int=1):
        # shape(in_dim,out_dim)
        # TODO:改batchsize
        in_dim, out_dim = shape
        # self.m_W=np.zeros(shape) 
        # self.m_b=np.zeros((1,out_dim))
        self.m_W = np.random.normal(loc=0.0, scale=0.01, size=shape)
        self.m_b = np.random.normal(loc=0.0, scale=0.01, size=(1, out_dim))
        self._forward_in = np.zeros((1,in_dim))
        self._forward_Z = np.zeros_like(self.m_b)
        self._forward_A = np.zeros_like(self.m_b)
        self.m_shape=shape
        self.m_batch_size=batch_size
        
    def forward(self,vector_input:np.ndarray)->np.ndarray:
        in_dim, out_dim = self.m_shape
        assert np.shape(vector_input)==(1,in_dim) #一定要是维度匹配的行向量
        self._forward_in = vector_input
        self._forward_Z = np.matmul(vector_input,self.m_W)+ self.m_b
        self._forward_A = self.m_activation_func(self._forward_Z)
        return self._forward_A

    def backward(self,grad_input:np.ndarray,learning_rate:float=0.01) ->np.ndarray:
        in_dim, out_dim = self.m_shape
        assert np.shape(grad_input)==(1,out_dim)
        dZ = grad_input * self.m_activation_func.gradient(self._forward_Z)
        dW = np.matmul(self._forward_in.T,dZ) / self.m_batch_size
        db = np.sum(dZ,axis=0,keepdims=True)/ self.m_batch_size
        grad_out = np.matmul(dZ,self.m_W.T)
        self.m_W -= learning_rate * dW
        self.m_b -= learning_rate * db
        return grad_out



In [60]:
class Rbf_layer(layer):
    input_dim = 84
    bitmap_list:np.ndarray
    _forward_in:np.ndarray
    _forward_out:np.ndarray
    def __init__(self):
        self.bitmap_list = rbf_bitmap()  #已经导入了10个rbf位图
        self._forward_in = np.zeros((1,self.input_dim))
        self._forward_out = np.zeros((1,len(self.bitmap_list)))

        
    def forward(self,vector_input:np.ndarray)->np.ndarray:
        distances = []
        self._forward_in = vector_input
        for bitmap in self.bitmap_list:
            dist = np.linalg.norm(vector_input - bitmap)
            distances.append(dist)
        return np.array(distances).reshape(1, -1)

    def backward(self, grad_input:np.ndarray, learning_rate: float) :
        # 只是为了满足该死的接口而已,grad_input固定输入正确的rbf参数向量
        return 2*(self._forward_in - grad_input).reshape(1,-1)
        
    def get_rbf_loss(self,correct_label):
        """
        反向传播的起点，计算rbf对f6输出a6的loss梯度
        """
        correct_rbf_param_vec =self.bitmap_list[correct_label]
        return self.backward(correct_rbf_param_vec,0)
        

In [64]:

f6 = Fc_layer((120,84),batch_size=1)
output_layer = Rbf_layer()


In [63]:

x_test = np.random.normal(0, 0.1, (1, 120))  # 输入到fc6的测试向量
target_label = 0  # 想匹配的rbf位图索引

# 简单的训练步骤
lr = 0.01
for step in range(100):
    # 前向
    fc_out = f6.forward(x_test)
    rbf_dist = output_layer.forward(fc_out)
    
    # 反向（从rbf层往fc层传梯度）
    rbf_grad = output_layer.get_rbf_loss(target_label)
    fc_grad = f6.backward(rbf_grad, learning_rate=lr)
    
    # 打印距离，看是不是越来越小
    if step % 10 == 0:
        print(f"Step {step}, 距离目标位图: {rbf_dist[0, target_label]:.4f}")

Step 0, 距离目标位图: 5.2951
Step 10, 距离目标位图: 3.2350
Step 20, 距离目标位图: 2.1322
Step 30, 距离目标位图: 1.4747
Step 40, 距离目标位图: 1.0522
Step 50, 距离目标位图: 0.7664
Step 60, 距离目标位图: 0.5664
Step 70, 距离目标位图: 0.4229
Step 80, 距离目标位图: 0.3181
Step 90, 距离目标位图: 0.2406
